# 心思维 AI 扒谱助手｜Colab GPU Worker V1

每次 Colab Runtime 重新连接后，运行下面唯一的启动单元即可。它会挂载 Google Drive、读取 Colab Secrets、获取并校验当前发布的 Worker 源码包、恢复 Python 环境、检查 BS-RoFormer 与 CUDA，然后启动 Worker。

**首次使用前**：在 Colab 左侧「Secrets」中添加 `MUSICMASTER_WORKER_TOKEN`，并允许此 Notebook 读取；确认已分配 GPU Runtime（例如 Tesla T4）。本流程不会创建、启动或释放 GPU。

**持久文件**：BS-RoFormer 权重保存在 `MyDrive/XinsW_Models/BS-RoFormer/`；首次运行会把 CREPE 模型和 CUDA 分离环境精确锁文件保存到 Drive。Python 虚拟环境仍在 Colab 临时磁盘，Runtime 重启后由脚本自动重建。

In [ ]:
from google.colab import drive, userdata
from pathlib import Path, PurePosixPath
import hashlib, json, os, re, shutil, subprocess, sys, tarfile, tempfile, urllib.request

# 固定连接到当前开发版网站；不要把 Worker Token 写进 Notebook。
API = "https://xinsw-transcriber-dev.onrender.com"
WORKER_ID = "colab-t4"
MODEL_DIR = "/content/drive/MyDrive/XinsW_Models/BS-RoFormer"
CACHE_ROOT = Path("/content/xinsw-worker-bundles")

drive.mount("/content/drive", force_remount=False)
worker_token = globals().get("new_worker_token") or userdata.get("MUSICMASTER_WORKER_TOKEN")
if not worker_token:
    raise RuntimeError("请先在 Colab Secrets 中添加 MUSICMASTER_WORKER_TOKEN，并授权本 Notebook 读取。")

os.environ["MUSICMASTER_API_URL"] = API
os.environ["MUSICMASTER_WORKER_TOKEN"] = worker_token
os.environ["MUSICMASTER_WORKER_ID"] = WORKER_ID
os.environ["MUSICMASTER_SEP_MODELS"] = MODEL_DIR
os.environ["MUSICMASTER_SEP_PYTHON"] = "/content/xinsw-sep-venv/bin/python"

# 从网站取得当前固定版本源码包清单，不在 Notebook 中写死 commit 或密钥。
req = urllib.request.Request(
    API + "/api/internal/worker/bundle",
    headers={"Authorization": "Bearer " + worker_token, "Accept": "application/json"},
)
with urllib.request.urlopen(req, timeout=30) as response:
    manifest = json.loads(response.read().decode("utf-8"))
commit = str(manifest.get("repository_commit", "")).lower()
digest = str(manifest.get("sha256", "")).lower()
bundle_url = str(manifest.get("url", ""))
if not re.fullmatch(r"[0-9a-f]{40}", commit):
    raise RuntimeError("网站返回的 Worker commit SHA 无效。")
if not re.fullmatch(r"[0-9a-f]{64}", digest):
    raise RuntimeError("网站返回的 Worker SHA-256 无效。")
if not bundle_url.startswith("https://"):
    raise RuntimeError("源码包下载地址不是 HTTPS。")

target = CACHE_ROOT / commit
marker = target / ".xinsw-bundle-sha256"
if marker.is_file() and marker.read_text(encoding="ascii").strip() == digest:
    print(f"源码包已缓存并验证：{commit[:12]}")
else:
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="xinsw-bootstrap-") as tmp:
        tmp = Path(tmp)
        archive_path = tmp / "worker.tar.gz"
        download_req = urllib.request.Request(bundle_url, headers={"User-Agent": "XinsW-Colab-Worker/1"})
        with urllib.request.urlopen(download_req, timeout=600) as response, archive_path.open("wb") as out:
            shutil.copyfileobj(response, out, length=1024 * 1024)
        hasher = hashlib.sha256()
        with archive_path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                hasher.update(chunk)
        if hasher.hexdigest() != digest:
            raise RuntimeError("Worker 源码包 SHA-256 校验失败，已停止启动。")

        staging = tmp / "source"
        staging.mkdir()
        expanded = 0
        with tarfile.open(archive_path, "r:gz") as archive:
            members = archive.getmembers()
            roots = {PurePosixPath(m.name).parts[0] for m in members if PurePosixPath(m.name).parts}
            if len(roots) != 1:
                raise RuntimeError("源码包目录结构异常。")
            for member in members:
                p = PurePosixPath(member.name)
                if p.is_absolute() or ".." in p.parts:
                    raise RuntimeError("源码包包含不安全路径。")
                parts = p.parts[1:]
                if not parts:
                    continue
                dest = staging.joinpath(*parts)
                if member.isdir():
                    dest.mkdir(parents=True, exist_ok=True)
                elif member.isfile():
                    expanded += int(member.size)
                    if expanded > 1024 * 1024 * 1024:
                        raise RuntimeError("源码包展开超过 1 GiB 限制。")
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    src = archive.extractfile(member)
                    if src is None:
                        raise RuntimeError("源码包文件读取失败。")
                    with src, dest.open("wb") as out:
                        shutil.copyfileobj(src, out, length=1024 * 1024)
                    dest.chmod(member.mode & 0o755)
                else:
                    raise RuntimeError("源码包包含链接或特殊文件，已拒绝解压。")
        if not (staging / "scripts/bootstrap_colab_worker.py").is_file():
            raise RuntimeError("源码包缺少 Colab 初始化脚本。")
        (staging / ".xinsw-bundle-sha256").write_text(digest, encoding="ascii")
        if target.exists():
            shutil.rmtree(target)
        staging.rename(target)
    print(f"源码包下载、SHA-256 校验和安全解压完成：{commit[:12]}")

required = [
    Path(MODEL_DIR) / "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    Path(MODEL_DIR) / "model_bs_roformer_ep_317_sdr_12.9755.yaml",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError("Google Drive 中缺少 BS-RoFormer 模型文件：" + "；".join(missing))

print("模型文件已确认。开始恢复/检查环境并启动 GPU Worker；此单元会持续运行。")
subprocess.run(
    [sys.executable, str(target / "scripts/bootstrap_colab_worker.py"), "--start"],
    cwd=str(target), env=os.environ.copy(), check=True,
)
